# TSMC 月营收加速度：Stage1 现金相对 SOXX 五日信息门

原 Stage1 39 项匹配对照；SEC 原报的保守次日时钟；2016–2022 训练与完整 2023–2026 日历复用诊断。


In [ ]:
import hashlib,json,os,pathlib,shlex
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from src.analysis.seed_disagreement_information import full_calendar_bootstrap
from src.data.nas_market_data import archive_nas_recorder,_settings,_ssh
root=pathlib.Path.cwd();card=root/'research/stage1_tsmc_monthly_cash_h5_information_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest();assert plan['frozen'] and plan['execution_authorized_by_active_Goal'] and not (card/'run_state.json').exists()
parent=root/src['parent_card'];source=root/src['source_retention_card'];provider=root/src['tsmc_provider_uri'];prec=parent/'qlib_recorder'/src['parent_experiment_id']/src['parent_recorder_id']/'artifacts'
for p,h in [(parent/'summary.json',src['parent_summary_sha256']),(prec/'inputs/features',src['parent_features_sha256']),(prec/'inputs/daily_frame',src['parent_frame_sha256']),(prec/'inputs/timing',src['parent_timing_sha256']),(prec/'models/base_outer_71',src['parent_model_sha256']),(source/'summary.json',src['source_retention_summary_sha256']),(provider/'qlib_store_manifest.json',src['tsmc_provider_manifest_sha256']),(root/'src/analysis/seed_disagreement_information.py',src['bootstrap_module_sha256'])]:assert sha(p)==h,(p,sha(p),h)
assert json.loads((source/'summary.json').read_text())['source_archive']['sha256']==json.loads((source/'nas_scoped_restore.json').read_text())['source_archive_sha256']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices'),region='us',kernels=1)
cal=pd.DatetimeIndex(D.calendar(start_time='2011-01-03',end_time='2026-10-09',freq='day'));assert len(cal)==3966
with R.uri_context(str(parent/'qlib_recorder')):
 rec=R.get_recorder(experiment_id=src['parent_experiment_id'],recorder_id=src['parent_recorder_id']);assert rec.status=='FINISHED';features=rec.load_object('inputs/features');frame=rec.load_object('inputs/daily_frame');timing=rec.load_object('inputs/timing');original=rec.load_object('models/base_outer_71')
assert frame.index.equals(features.index) and frame.index.equals(timing.index);base=list(original.base_features);assert len(base)==39 and set(base).issubset(features.columns)
qlib.init(provider_uri=str(provider),region='us',kernels=1)
raw=D.features(['TSMC_RELEASE'],['$acceleration_pct_points','$event_available'],start_time='2015-12-11',end_time='2026-09-25',freq='day');assert raw.index.get_level_values('instrument').unique().tolist()==['TSMC_RELEASE'];source_frame=raw.droplevel('instrument').reindex(frame.index)
source_known=source_frame['$acceleration_pct_points'].notna();maturity=pd.to_datetime(timing.return_available_at.shift(-4),utc=True)
parts=[frame.soxx_forward_open_return.astype(float).shift(-i) for i in range(5)];valid=np.logical_and.reduce([np.isfinite(x.to_numpy(float))&(x.to_numpy(float)>-1) for x in parts]);cash_target=(-np.log1p(pd.concat(parts,axis=1)).sum(axis=1)).where(valid)
train_dates=frame.index.to_series().between(*plan['cohorts']['train_signals']).to_numpy();held_dates=frame.index.to_series().between(*plan['cohorts']['held_signals']).to_numpy();cut=pd.Timestamp('2023-01-01',tz='America/New_York').tz_convert('UTC');terminal=pd.Timestamp('2026-10-10',tz='America/New_York').tz_convert('UTC')
train_mask=train_dates&source_known.to_numpy()&maturity.lt(cut).to_numpy()&cash_target.notna().to_numpy();held_mask=held_dates&source_known.to_numpy()&maturity.lt(terminal).to_numpy()&cash_target.notna().to_numpy()
coh=plan['cohorts'];assert train_dates.sum()==coh['train_calendar'] and train_mask.sum()==coh['train_mature_known'] and held_dates.sum()==coh['held_full_calendar'] and held_mask.sum()==coh['held_mature_source_known'] and (held_dates&~source_known.to_numpy()).sum()==coh['held_unknown_tail']
assert int(source_frame.loc[train_dates,'$event_available'].fillna(0).sum())==coh['train_source_events'] and int(source_frame.loc[held_dates,'$event_available'].fillna(0).sum())==coh['held_source_events_through_2026_09_25']
extra='tsmc_monthly_yoy_acceleration_pct_points';f=features[base].copy();f[extra]=source_frame['$acceleration_pct_points'].to_numpy(float);meta=pd.DataFrame({'train_mature_source_known':train_mask,'held_mature_source_known':held_mask,'label_available_at':maturity,'source_known':source_known.to_numpy(bool),'source_event':source_frame['$event_available'].fillna(0).to_numpy(float)},index=frame.index);static=pd.concat({'feature':f,'label':pd.DataFrame({'cash_minus_SOXX_H5':cash_target},index=frame.index),'meta':meta},axis=1);static.index=pd.MultiIndex.from_arrays([frame.index,['SOXL_SOXX_CASH']*len(frame)],names=['datetime','instrument']);dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':tuple(coh['train_signals']),'held':tuple(coh['held_signals'])})
tr=dataset.prepare('train',col_set=['feature','label','meta']);he=dataset.prepare('held',col_set=['feature','label','meta']);train=tr.loc[tr[('meta','train_mature_source_known')].astype(bool)];held=he.loc[he[('meta','held_mature_source_known')].astype(bool)];assert len(tr)==1757 and len(he)==944 and len(train)==1751 and len(held)==936
held_dates_scored=pd.DatetimeIndex(held.index.get_level_values('datetime'));maxcorr=float(features.loc[frame.index[train_mask],base].corrwith(source_frame.loc[train_mask,'$acceleration_pct_points'],method='spearman').abs().max());assert maxcorr<0.8
preflight={'status':'pinned_original39_TSMC_source_and_H5_cash_cohorts_matched','plan_sha256':sha(card/'plan.json'),'train_calendar':len(tr),'train_mature':len(train),'held_calendar':len(he),'held_scored':len(held),'held_unknown_tail':8,'train_source_events':84,'held_source_events':45,'max_abs_train_original39_feature_Spearman':maxcorr,'source_Recorder_object_loads':4,'model_fits':0,'native_accounts':0};(card/'preflight_result.json').write_text(json.dumps(preflight,indent=2)+'\n')
state={'status':'fit_started','plan_sha256':preflight['plan_sha256'],'fit_attempts':0,'fit_completed':0,'bootstrap_attempts':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
x0=train['feature'][base].to_numpy(float);xh0=held['feature'][base].to_numpy(float);x1=train['feature'][base+[extra]].to_numpy(float);xh1=held['feature'][base+[extra]].to_numpy(float);y=train[('label','cash_minus_SOXX_H5')].to_numpy(float);yh=held[('label','cash_minus_SOXX_H5')].to_numpy(float);assert np.isfinite(y).all() and np.isfinite(yh).all() and np.isfinite(x1[:,-1]).all() and np.isfinite(xh1[:,-1]).all()
models=[]
for x in (x0,x1):
 model=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),Ridge(alpha=1.0));state['fit_attempts']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');model.fit(x,y);models.append(model);state['fit_completed']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
base_model,candidate=models;p0=base_model.predict(xh0);p1=candidate.predict(xh1);mean=float(y.mean());coef=float(candidate.named_steps['ridge'].coef_[-1]);scored=pd.DataFrame({'signal_date':held_dates_scored,'label':yh,'baseline_prediction':p0,'candidate_prediction':p1,'train_mean_prediction':mean});scored['gain_vs_baseline']=(yh-p0)**2-(yh-p1)**2;scored['gain_vs_train_mean']=(yh-mean)**2-(yh-p1)**2
full=pd.DataFrame({'signal_date':pd.DatetimeIndex(he.index.get_level_values('datetime'))}).merge(scored,on='signal_date',how='left',validate='one_to_one');assert len(full)==944 and full.label.notna().sum()==936
metrics={'candidate_MSE':float(np.square(yh-p1).mean()),'baseline_MSE':float(np.square(yh-p0).mean()),'train_mean_MSE':float(np.square(yh-mean).mean()),'added_standardized_coefficient':coef,'comparators':{}};gates={'coefficient_negative':bool(coef<0)};reps={};indices=None;boot=plan['information_gate']['bootstrap']
for key in ('baseline','train_mean'):
 diff=scored['gain_vs_'+key];half=[]
 for pos in (slice(0,472),slice(472,944)):
  subset=full.iloc[pos];half.append({'first_date':str(subset.signal_date.iloc[0].date()),'last_date':str(subset.signal_date.iloc[-1].date()),'calendar_rows':len(subset),'scored_rows':int(subset.label.notna().sum()),'mean_gain':float(subset['gain_vs_'+key].mean())})
 annual=[]
 for year in (2024,2025):
  subset=scored.loc[scored.signal_date.dt.year.eq(year)];annual.append({'year':year,'rows':len(subset),'mean_gain':float(subset['gain_vs_'+key].mean())})
 state['bootstrap_attempts']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');interval,idx,rep=full_calendar_bootstrap(full['gain_vs_'+key].to_numpy(float),mean_block=boot['mean_block_sessions'],resamples=boot['draws_each'],seed=boot['seed'],alpha=boot['one_sided_alpha_each']);assert interval['valid_replicates']==boot['draws_each']
 if indices is None:indices=idx
 else:assert np.array_equal(indices,idx)
 reps[key]=rep;metrics['comparators'][key]={'full_mean_gain':float(diff.mean()),'halves':half,'complete_years':annual,'bootstrap':interval};gates[key]={'full_positive':bool(diff.mean()>0),'both_halves_positive':bool(all(z['mean_gain']>0 for z in half)),'both_2024_2025_positive':bool(all(z['mean_gain']>0 for z in annual)),'corrected_lower_positive':bool(interval['lower_bound']>0)}
passed=bool(gates['coefficient_negative'] and all(all(v.values()) for v in gates.values() if isinstance(v,dict)))
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();dataset.config(dump_all=True,recursive=True);rec.save_objects(artifact_path='inputs',plan=plan,preflight=preflight,dataset=dataset);rec.save_objects(artifact_path='models',baseline=base_model,candidate=candidate);rec.save_objects(artifact_path='predictions',scored=scored);rec.save_objects(artifact_path='analysis',full_calendar=full,metrics=metrics,gates=gates,bootstrap_replicates=reps,bootstrap_indices=indices,information_gate_pass=passed);rec.log_metrics(fits=2,native_accounts=0,information_gate_pass=int(passed),gain_vs_baseline=metrics['comparators']['baseline']['full_mean_gain']);rid,eid=rec.id,str(rec.experiment_id)
state['status']='finished';(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':'conditional_TSMC_monthly_cash_H5_information_pass_needs_0bp_account' if passed else 'not_promoted_fixed_TSMC_monthly_cash_H5_information_failed','objective_achieved':False,'plan_sha256':preflight['plan_sha256'],'preflight':preflight,'metrics':metrics,'gates':gates,'information_gate_pass':passed,'new_model_fits':2,'new_bootstraps':2,'new_native_accounts':0,'historical_independence':False,'recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','models/*','predictions/*','analysis/*']},'limitations':['Conservative SEC acceptance-next-US-session is later than possible issuer first press release.','Eight held tail sessions are unknown in the fixed TSMC provider; all 944 calendar dates remain in bootstrap.','Information MSE does not prove a funded Stage1 SOXX excess or independent forward validation.']};(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid);remote=_ssh(_settings(root)['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0];assert remote==evidence['sha256'];summary['NAS_evidence']=evidence;summary['NAS_readback']={'sha256':remote,'verified':True};(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({'decision':summary['decision'],'preflight':preflight,'metrics':metrics,'gates':gates,'recorder_id':rid,'NAS_sha256':remote},ensure_ascii=False,indent=2))


## 中文结论

固定原 Stage1 **39 项输入**，只在两臂相同的2016–2022已成熟、TSMC来源已知 **1,751** 日拟合两次 Ridge；候选额外加入一项 SEC 原报的月营收同比加速度。保守可用时点是 SEC 接收后下一美股交易日。2023–2026完整评价日历 **944** 日中，**936** 日有成熟标签和来源；末端8日按未知保留，区块检验仍沿完整日历抽样。训练/评价期间分别有84/45个原报更新，936日持有值不能当作936次独立月度消息。原始来源264文件、旧来源Recorder和本卡Recorder均已 NAS 冷读。

新增标准化系数 **+0.00377166**，与事前“需求加速应降低现金相对 SOXX 价值”的负号假设相反。候选评价 MSE **0.002599901**，原39匹配模型 **0.002608944**，训练均值 **0.002342643**。相对原39，候选完整平均配对改善仅 **+9.042848904e-06**，校正区块下界 **-5.515401671e-05**；相对训练均值则恶化 **-2.572582160e-04**，下界 **-5.402590992e-04**。

失败区间对比保留全部15季度和四个年度：相对原39，完整2024/2025的平均MSE改善为 **+3.733730748e-05／+3.110309307e-05**，2023/2026部分年度却为 **-2.187377105e-05／-1.767513812e-05**；候选对训练均值在两完整年度也都更差。这是重复使用历史的误差归因，不给按年份切换的交易规则。

决定 **`not_promoted_fixed_TSMC_monthly_cash_H5_information_failed`**。固定符号、五日任务、训练段和对照不再救援；停止在信息门，**2次拟合、2次区块检验、0新资金账户**。Qlib Recorder `6ffc6344cbb84600b608da12f1640844`、NAS SHA-256 `c3c14673b6785ae8dfa36369187d21e741f50501c609b691d25a7b50c13c5b1b` 的模型／预测／门对象冷读一致。信息误差不能替代完整 Stage1 0bp Tiered 账户的 SOXX 财富、回撤、年度、126日窗口及置信门；2/5bp和独立前向证据仍缺。
